In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import ClusterCentroids
from imblearn.combine import SMOTEENN

file_path = Path('LoanStats_2019Q1.csv')
df = pd.read_csv(file_path, low_memory=False)
if 'loan_status' not in df.columns:
    df = pd.read_csv(file_path, skiprows=1, low_memory=False)
status_mapping = {'Fully Paid': 'low_risk', 'Current': 'low_risk', 'Charged Off': 'high_risk', 'Late (31-120 days)': 'high_risk', 'Late (16-30 days)': 'high_risk', 'Default': 'high_risk', 'In Grace Period': 'high_risk'}
df = df.loc[df['loan_status'].isin(status_mapping)].copy()
y = df['loan_status'].map(status_mapping)
feature_columns = ['loan_amnt', 'int_rate', 'installment', 'home_ownership', 'annual_inc', 'verification_status', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'total_acc', 'initial_list_status', 'application_type']
X = df.loc[:, [column for column in feature_columns if column in df.columns]].copy()
if X.empty or y.nunique() != 2:
    raise ValueError('The dataset must contain features and both target classes.')
if 'int_rate' in X:
    X['int_rate'] = pd.to_numeric(X['int_rate'].astype(str).str.rstrip('%'), errors='coerce') / 100
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=1, test_size=0.25)
usable_columns = X_train.columns[X_train.notna().any()].tolist()
X_train, X_test = X_train[usable_columns], X_test[usable_columns]


In [ ]:
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = X_train.columns.difference(categorical_columns).tolist()
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', encoder)]), categorical_columns)
])
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
print(y_train.value_counts())


In [ ]:
minority_count = y_train.value_counts().min()
if minority_count < 2:
    raise ValueError('At least two training samples per class are required.')
k_neighbors = min(5, minority_count - 1)
resamplers = {
    'Random Oversampling': RandomOverSampler(random_state=1),
    'SMOTE': SMOTE(random_state=1, k_neighbors=k_neighbors),
    'Cluster Centroids': ClusterCentroids(random_state=1),
    'SMOTEENN': SMOTEENN(random_state=1, smote=SMOTE(random_state=1, k_neighbors=k_neighbors))
}
results = []
for name, resampler in resamplers.items():
    X_resampled, y_resampled = resampler.fit_resample(X_train_encoded, y_train)
    if pd.Series(y_resampled).nunique() != 2:
        raise ValueError(f'{name} must retain both classes.')
    model = LogisticRegression(random_state=1, max_iter=2000)
    model.fit(X_resampled, y_resampled)
    predictions = model.predict(X_test_encoded)
    score = balanced_accuracy_score(y_test, predictions)
    results.append({'Method': name, 'Balanced Accuracy': score})
    print(name)
    print(confusion_matrix(y_test, predictions, labels=['high_risk', 'low_risk']))
    print(classification_report(y_test, predictions, zero_division=0))
pd.DataFrame(results)
